# Lesson 2: Tool Calling

## Setup

In [ ]:
import os 
import sys
sys.path.append("/home/as/Documents/GitHub/Llamaindex_Practice")


In [14]:
from helper import get_openai_api_key
os.environ["OPENAI_API_KEY"] = get_openai_api_key()
print("Key loaded successfully!")

Key loaded successfully!


In [31]:
from llama_index.core import Settings
from llama_index.llms.openai import OpenAI

Settings.llm = OpenAI(model="gpt-4o-mini")

In [16]:
import nest_asyncio
nest_asyncio.apply()

## 1. Define a Simple Tool

In [17]:
from llama_index.core.tools import FunctionTool

def add(x: int, y: int) -> int:
    """Adds two integers together."""
    return x + y

def mystery(x: int, y: int) -> int: 
    """Mystery function that operates on top of two numbers."""
    return (x + y) * (x + y)


add_tool = FunctionTool.from_defaults(fn=add)
mystery_tool = FunctionTool.from_defaults(fn=mystery)

In [ ]:
llm = OpenAI(model="gpt-4o-mini")
response = llm.predict_and_call(
    [add_tool, mystery_tool], 
    "Tell me the output of the mystery function on 2 and 9", 
    verbose=True
)
print(str(response))

=== Calling Function ===
Calling function: mystery with args: {"x": 2, "y": 9}
=== Function Output ===
121
121


## 2. Define an Auto-Retrieval Tool

### Load Data

In [19]:
from llama_index.core import SimpleDirectoryReader
# load documents
documents = SimpleDirectoryReader(input_files=["/home/as/Documents/GitHub/Llamaindex_Practice/metagpt.pdf"]).load_data()

In [20]:
from llama_index.core.node_parser import SentenceSplitter
splitter = SentenceSplitter(chunk_size=1024)
nodes = splitter.get_nodes_from_documents(documents)

In [21]:
print(nodes[0].get_content(metadata_mode="all"))

page_label: 1
file_name: metagpt.pdf
file_path: /home/as/Documents/GitHub/Llamaindex_Practice/metagpt.pdf
file_type: application/pdf
file_size: 3664341
creation_date: 2026-11-16
last_modified_date: 2026-11-13

Preprint
METAGPT: M ETAPROGRAMMING FOR A
MULTI-AGENT COLLABORATIVE FRAMEWORK
Sirui Hong1∗, Mingchen Zhuge2∗, Jonathan Chen1, Xiawu Zheng3, Yuheng Cheng4,
Ceyao Zhang4,Jinlin Wang1,Zili Wang,Steven Ka Shing Yau5,Zijuan Lin4,
Liyang Zhou6,Chenyu Ran1,Lingfeng Xiao1,7,Chenglin Wu1†,J¨urgen Schmidhuber2,8
1DeepWisdom,2AI Initiative, King Abdullah University of Science and Technology,
3Xiamen University,4The Chinese University of Hong Kong, Shenzhen,
5Nanjing University,6University of Pennsylvania,
7University of California, Berkeley,8The Swiss AI Lab IDSIA/USI/SUPSI
ABSTRACT
Remarkable progress has been made on automated problem solving through so-
cieties of agents based on large language models (LLMs). Existing LLM-based
multi-agent systems can already solve simple dialogue tasks. 

In [22]:
from llama_index.core import VectorStoreIndex

vector_index = VectorStoreIndex(nodes)
query_engine = vector_index.as_query_engine(similarity_top_k=2)

In [24]:
from llama_index.core.vector_stores import MetadataFilters

query_engine = vector_index.as_query_engine(
    similarity_top_k=2,
    filters=MetadataFilters.from_dicts(
        [
            {"key": "page_label", "value": "2"}
        ]
    ),
    llm = OpenAI(model="gpt-4o-mini")
)

response = query_engine.query(
    "What are some high-level results of MetaGPT?", 
)

In [25]:
print(str(response))

MetaGPT achieves notable high-level results in code generation benchmarks, attaining a state-of-the-art Pass@1 rate of 85.9% and 87.7% on the HumanEval and MBPP evaluations, respectively. Additionally, it demonstrates a 100% task completion rate in experimental evaluations, showcasing its robustness and efficiency in handling complex software projects.


In [26]:
for n in response.source_nodes:
    print(n.metadata)

{'page_label': '2', 'file_name': 'metagpt.pdf', 'file_path': '/home/as/Documents/GitHub/Llamaindex_Practice/metagpt.pdf', 'file_type': 'application/pdf', 'file_size': 3664341, 'creation_date': '2026-11-16', 'last_modified_date': '2026-11-13'}


### Define the Auto-Retrieval Tool

In [27]:
from typing import List
from llama_index.core.vector_stores import FilterCondition


def vector_query(
    query: str, 
    page_numbers: List[str]
) -> str:
    """Perform a vector search over an index.
    
    query (str): the string query to be embedded.
    page_numbers (List[str]): Filter by set of pages. Leave BLANK if we want to perform a vector search
        over all pages. Otherwise, filter by the set of specified pages.
    
    """

    metadata_dicts = [
        {"key": "page_label", "value": p} for p in page_numbers
    ]
    
    query_engine = vector_index.as_query_engine(
        similarity_top_k=2,
        filters=MetadataFilters.from_dicts(
            metadata_dicts,
            condition=FilterCondition.OR
        )
    )
    response = query_engine.query(query)
    return response
    

vector_query_tool = FunctionTool.from_defaults(
    name="vector_tool",
    fn=vector_query
)

In [32]:
llm = OpenAI(model="gpt-4o-mini", temperature=0)
response = llm.predict_and_call(
    [vector_query_tool], 
    "What are the high-level results of MetaGPT as described on page 2?", 
    verbose=True
)

=== Calling Function ===
Calling function: vector_tool with args: {"query": "high-level results of MetaGPT", "page_numbers": ["2"]}
=== Function Output ===
MetaGPT achieves a new state-of-the-art performance in code generation benchmarks, with Pass@1 rates of 85.9% and 87.7% on the HumanEval and MBPP datasets, respectively. It demonstrates a 100% task completion rate in experimental evaluations, showcasing its robustness and efficiency in handling complex software projects. Additionally, it outperforms other frameworks like AutoGPT, LangChain, AgentVerse, and ChatDev, particularly in managing higher levels of software complexity and providing extensive functionality.


In [33]:
for n in response.source_nodes:
    print(n.metadata)

{'page_label': '2', 'file_name': 'metagpt.pdf', 'file_path': '/home/as/Documents/GitHub/Llamaindex_Practice/metagpt.pdf', 'file_type': 'application/pdf', 'file_size': 3664341, 'creation_date': '2026-11-16', 'last_modified_date': '2026-11-13'}


## Add some other tools

In [34]:
from llama_index.core import SummaryIndex
from llama_index.core.tools import QueryEngineTool

summary_index = SummaryIndex(nodes)
summary_query_engine = summary_index.as_query_engine(
    response_mode="tree_summarize",
    use_async=True,
)
summary_tool = QueryEngineTool.from_defaults(
    name="summary_tool",
    query_engine=summary_query_engine,
    description=(
        "Useful if you want to get a summary of MetaGPT"
    ),
)

In [35]:
response = llm.predict_and_call(
    [vector_query_tool, summary_tool], 
    "What are the MetaGPT comparisons with ChatDev described on page 8?", 
    verbose=True
)

=== Calling Function ===
Calling function: vector_tool with args: {"query": "MetaGPT comparisons with ChatDev", "page_numbers": ["8"]}
=== Function Output ===
MetaGPT demonstrates superior performance compared to ChatDev across various metrics in the SoftwareDev dataset. In terms of executability, MetaGPT scores 3.75, approaching a flawless score of 4, while ChatDev only scores 2.25. Additionally, MetaGPT requires less running time, taking 503 seconds compared to ChatDev's 762 seconds. 

When it comes to token usage, MetaGPT uses more tokens overall (24,613 or 31,255) compared to ChatDev's 19,292, but it generates code more efficiently, needing only 126.5/124.3 tokens per line of code, whereas ChatDev requires 248.9 tokens. 

In terms of code statistics, MetaGPT produces more code files (4.6 for MetaGPT compared to 1.9 for ChatDev) and a greater total number of code lines (194.6 for MetaGPT versus 77.5 for ChatDev). Furthermore, the human revision cost is significantly lower for MetaGP

In [36]:
for n in response.source_nodes:
    print(n.metadata)

{'page_label': '8', 'file_name': 'metagpt.pdf', 'file_path': '/home/as/Documents/GitHub/Llamaindex_Practice/metagpt.pdf', 'file_type': 'application/pdf', 'file_size': 3664341, 'creation_date': '2026-11-16', 'last_modified_date': '2026-11-13'}


In [40]:
response = llm.predict_and_call(
    [vector_query_tool, summary_tool], 
    "What is a summary of the paper Metagpt?", 
    verbose=True,
    error_on_no_tool_call=False 
)

=== Calling Function ===
Calling function: summary_tool with args: {"input": "Metagpt"}
=== Function Output ===
MetaGPT is a meta-programming framework designed for multi-agent collaboration utilizing large language models (LLMs). It enhances automated problem-solving by incorporating Standardized Operating Procedures (SOPs) into the workflow of LLM-based multi-agent systems. This framework allows for the decomposition of complex tasks into manageable subtasks, with specialized agents assigned specific roles such as Product Manager, Architect, Engineer, and QA Engineer. 

MetaGPT employs structured communication protocols to improve the efficiency of interactions among agents, utilizing a shared message pool and a subscription mechanism to filter relevant information. Additionally, it features an executable feedback mechanism that enables agents to iteratively improve code quality during runtime. 

In experimental evaluations, MetaGPT has demonstrated state-of-the-art performance on va

In [41]:
print(str(response))

MetaGPT is a meta-programming framework designed for multi-agent collaboration utilizing large language models (LLMs). It enhances automated problem-solving by incorporating Standardized Operating Procedures (SOPs) into the workflow of LLM-based multi-agent systems. This framework allows for the decomposition of complex tasks into manageable subtasks, with specialized agents assigned specific roles such as Product Manager, Architect, Engineer, and QA Engineer. 

MetaGPT employs structured communication protocols to improve the efficiency of interactions among agents, utilizing a shared message pool and a subscription mechanism to filter relevant information. Additionally, it features an executable feedback mechanism that enables agents to iteratively improve code quality during runtime. 

In experimental evaluations, MetaGPT has demonstrated state-of-the-art performance on various benchmarks, achieving high success rates in code generation tasks and showcasing its ability to handle com